# Garment Worker Productivity - Model Comparison

This notebook compares different regression models for predicting garment team productivity.

The same data preparation, chronological train/test split and evaluation metrics from the previous milestone are kept so that the models can be compared fairly.

In [2]:
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

## 1. Loading and Preparing the Data

In this part the dataset is loaded and prepared using the same cleaning and chronological train/test split used in the previous notebook-02. Keeping the same split means that each model is evaluated on the same unseen test data.

In [3]:
# Loads the dataset
df = pd.read_csv("../data/garments_worker_productivity.csv")

df["department"] = df["department"].str.strip()  # Cleans the department labels
df["date"] = pd.to_datetime(df["date"])  # Converst the date column

df = df.sort_values("date").reset_index(drop=True)  # Sorts the data by date

df.head()

,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,2015-01-01,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,2015-01-01,Quarter1,finishing,Thursday,11,0.70,4.15,NaN,1440,0,0.0,0,0,12.0,0.436326
2,2015-01-01,Quarter1,sweing,Thursday,4,0.65,23.69,861.0,7200,0,0.0,0,0,60.0,0.521180
3,2015-01-01,Quarter1,finishing,Thursday,7,0.80,2.90,NaN,960,0,0.0,0,0,8.0,0.540729
4,2015-01-01,Quarter1,finishing,Thursday,4,0.75,3.94,NaN,2160,0,0.0,0,0,18.0,0.593056


### 1.1 Features, Target and Train/Test Split

In [ ]:
# Separates the model inputs from the target
X = df.drop(columns=["actual_productivity", "date"])
y = df["actual_productivity"]

# Finds the unique dates in chronological order
unique_dates = df["date"].sort_values().unique()

# It uses approximately the first 80% of dates for training
split_index = int(len(unique_dates) * 0.8)
cutoff_date = unique_dates[split_index]

# Creating the training and test sets using whole dates
train_mask = df["date"] < cutoff_date
test_mask = df["date"] >= cutoff_date

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]
y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((943, 13), (254, 13), (943,), (254,))

## 2. Preprocessing

The same preprocessing approach from the previous notebook-02 is used so that the learned models are trained on consistently prepared data.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [6]:
# Categorical features
categorical_features = [
    "quarter",
    "department",
    "day",
    "team"
]

# Numerical features
numerical_features = [
    "targeted_productivity",
    "smv",
    "over_time",
    "incentive",
    "idle_time",
    "idle_men",
    "no_of_style_change",
    "no_of_workers"
]

# WIP
wip_feature = ["wip"]

In [7]:
# Prepares the numerical features
numerical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Prepares WIP separately
wip_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
    ("scaler", StandardScaler())
])

# Converts categorical values into numerical columns
categorical_preprocessor = OneHotEncoder(
    handle_unknown="ignore"
)

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_preprocessor, numerical_features),
        ("wip", wip_preprocessor, wip_feature),
        ("cat", categorical_preprocessor, categorical_features)
    ]
)

## 3. Linear Regression

In my project, Linear Regression is used as the first learned model. Unlike the DummyRegressor, it uses the input features to learn relationships with `actual_productivity`.

The preprocessing pipeline is combined with the model so that the same preparation is applied to the training and test data.

In [9]:
# Combining preprocessing with Linear Regression
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

In [12]:
# Training the Linear Regression model
linear_model.fit(X_train, y_train);

In [14]:
# Predicts productivity for the unseen test data
y_pred_linear = linear_model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# Evaluates the Linear Regression predictions
linear_mae = mean_absolute_error(y_test, y_pred_linear)
linear_rmse = root_mean_squared_error(y_test, y_pred_linear)
linear_r2 = r2_score(y_test, y_pred_linear)

print(f"Linear Regression MAE: {linear_mae:.4f}")
print(f"Linear Regression RMSE: {linear_rmse:.4f}")
print(f"Linear Regression R²: {linear_r2:.4f}")

Linear Regression MAE: 0.2853
Linear Regression RMSE: 1.1537
Linear Regression R²: -49.8455


In [17]:
# Check the range of Linear Regression predictions
print("Lowest prediction:", y_pred_linear.min())
print("Highest prediction:", y_pred_linear.max())

Lowest prediction: 0.35134152186993434
Highest prediction: 12.673362930069903


In [18]:
# Compare incentive values in the training and test sets
print("Highest training incentive:", X_train["incentive"].max())
print("Highest test incentive:", X_test["incentive"].max())

Highest training incentive: 138
Highest test incentive: 3600


### Linear Regression Observations

As we can see the Linear Regression performed much worse than the baseline. Further checks showed that the test data contains incentive values much higher than those seen during training. The highest training incentive was 138, while the test set contained values up to 3600.

The model also produced an unrealistic maximum productivity prediction of approximately 12.67. This suggests that Linear Regression struggled to generalise when the test data contained values outside the range it had learned from.

## 4. Decision Tree Regressor

Next I used a Decision Tree Regressor as the second learned model. It makes predictions by splitting the training data into groups based on the input features.

Unlike Linear Regression, it can learn non-linear relationships between the features and productivity.